# Isolated students

Can you spot first-year students who are not part of any group of fellow students, without asking anyone?

Universities (should) care if first-year students fit in socially, since isolation is linked to dropping out.


## Setup

In [21]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
import preprocess as pp
import friends as fr
import isolated as iso


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
bt = pp.load_bt()
prox = pp.proximity()
st = pp.student_table()
fb = pp.clean_fb(pp.load_fb())


In [23]:
# parameters, change and rerun from here
rssi = -80
min_coverage = 0.5
k = 20 # neighbours for LOF
top_share = 0.05 # share of students to flag


## One row per student

Each student gets two numbers from the proximity data. "Together" means the same as in `friends`: a signal of -80 dBm or stronger outside of teaching hours. In a lecture the schedule decides who is near, so teaching hours don't say much about groups. `friends` shows this: of the pairs near each other at least once in class, 9.2% are Facebook friends, against 15.1% outside class.

| Feature | What it is |
|---|---|
| `company` | share of the hours outside class, with the phone on, when anyone was near |
| `partners` | number of different students near during the 28 days |

Only students with a coverage of 50% or more are used. A phone that is switched off looks the same as a student who is alone, so the other students are "not observed", not isolated. For the same reason, `company` counts only hours when the phone was on.

Facebook, calls and SMS are not used here. They are used later to check the result, and that check is only fair if the method never saw them.


In [24]:
observed = iso.observed_hours(bt)
# SELECT user FROM st WHERE coverage >= min_coverage
ids = st.index[st.coverage >= min_coverage]
f = iso.features(fr.together(prox, rssi), observed, ids)
print(f"{(st.coverage > 0).sum()} students with bluetooth data, {len(ids)} with coverage of {min_coverage:.0%} or more")
f.describe().round(3)


706 students with bluetooth data, 615 with coverage of 50% or more


,company,partners
count,615.000,615.000
mean,0.185,36.763
std,0.217,35.310
min,0.000,0.000
25%,0.031,11.000
50%,0.093,26.000
75%,0.245,50.000
max,0.925,181.000


## Never with anyone

Some students had their phone on at least half of the time but were never near other students outside of class. They are the clearest answer to the question.

They have the same values so to LOF they're a dense group and each of them gets the score 1.0, which means normal. scikit-learn also warns about this. The course calls it masking: anomalies that are close together hide each other. So these students are reported here, and LOF only gets the students who were near someone at least once.


In [25]:
# SELECT user FROM f WHERE company = 0
never = f.index[f.company == 0]
print(f"{len(never)} students never near another student outside class")
# lof with these students included, no log since log10(0) is not defined
# SELECT DISTINCT ROUND(lof, 3) FROM lof(f) WHERE user IN never
print("their LOF scores:", iso.lof(iso.scale(f, log=False), k)[never].round(3).unique())


23 students never near another student outside class
their LOF scores: [1.]


/home/tobias/skola/uu/dm1/data-mining-1/.venv/lib/python3.10/site-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
